In [2]:
import os
import contextlib
import joblib
from joblib import Parallel, delayed
from tqdm import tqdm

import numpy as np
from pathlib import Path
import tifffile as tiff

from skimage.filters import gaussian, laplace
from skimage.transform import resize

from liffile import LifFile

In [3]:

# -------------------------
# tqdm + joblib bridge
# -------------------------
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

def edge_band_mask(shape, frac=0.375):
    """Mask that is 1 in a border band and 0 in the interior."""
    H, W = shape
    bw = int(round(min(H, W) * frac))
    m = np.zeros((H, W), dtype=bool)
    m[:bw, :] = True
    m[-bw:, :] = True
    m[:, :bw] = True
    m[:, -bw:] = True
    return m

def focus_score_lapvar_roi(img2d, sigma_bg=10, mask=None):
    img = _to_float32(img2d)
    hp = img - gaussian(img, sigma=sigma_bg, preserve_range=True)
    L = laplace(hp, ksize=3)

    if mask is None:
        return float(np.var(L))

    vals = L[mask]
    if vals.size < 32:  # safety
        return float(np.var(L))
    return float(np.var(vals))
# -------------------------
# focus + downsample utils
# -------------------------
def _to_float32(img):
    img = img.astype(np.float32, copy=False)
    p1, p99 = np.percentile(img, (1, 99))
    if p99 > p1:
        img = (img - p1) / (p99 - p1)
    return np.clip(img, 0.0, 1.0)

def downsample_stack(stack_zyx, factor=4):
    Z, Y, X = stack_zyx.shape
    y2, x2 = Y // factor, X // factor
    out = np.empty((Z, y2, x2), dtype=np.float32)
    for z in range(Z):
        out[z] = resize(
            stack_zyx[z].astype(np.float32, copy=False),
            (y2, x2),
            order=1,
            preserve_range=True,
            anti_aliasing=True
        ).astype(np.float32)
    return out

def focus_score_lapvar(img2d, sigma_bg=10):
    img = _to_float32(img2d)
    hp = img - gaussian(img, sigma=sigma_bg, preserve_range=True)
    L = laplace(hp, ksize=3)
    return float(np.var(L))

def best_focus_z(stack_zyx_ds, sigma_bg=10, mask=None):
    Z = stack_zyx_ds.shape[0]
    scores = np.zeros(Z, dtype=np.float32)
    for z in range(Z):
        scores[z] = focus_score_lapvar_roi(stack_zyx_ds[z], sigma_bg=sigma_bg, mask=mask)
    return int(np.argmax(scores)), scores


# -------------------------
# one job = one LIF file
# -------------------------
def process_lif_file(
    lif_path: str,
    img_out_dir: str,
    lbl_out_dir: str,
    downsample: int = 4,
    sigma_bg_ds: float = 10,
    label_dtype: str = "uint8",
):
    lif_path = Path(lif_path)
    img_out_dir = Path(img_out_dir)
    lbl_out_dir = Path(lbl_out_dir)
    lbl_dtype = np.dtype(label_dtype)

    lif_name = lif_path.stem.lower().replace(".", "_")
    n_saved, n_skipped = 0, 0

    # optional step() import (only if you need pixel sizes)
    try:
        from your_module_with_step import step  # <-- edit or remove
        have_step = True
    except Exception:
        have_step = False

    lif = None
    try:
        lif = LifFile(str(lif_path))

        for i in range(len(lif.images)):
            try:
                img = lif.images[i]

                if img.dims == ('C', 'Z', 'Y', 'X'):
                    stack = img.asarray()[0]
                elif img.dims == ('Z', 'Y', 'X'):
                    stack = img.asarray()
                else:
                    n_skipped += 1
                    continue

                if stack.ndim != 3 or stack.shape[0] < 2:
                    n_skipped += 1
                    continue

                stack_ds = downsample_stack(stack, factor=downsample)

                z0, scores = best_focus_z(stack_ds, sigma_bg=sigma_bg_ds)
                best2d = stack_ds[z0].astype(np.float32)

                meta = {}
                if have_step:
                    try:
                        xa = img.asxarray()
                        x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
                        if x_um is not None:
                            x_um_ds = float(x_um * downsample)
                            meta = {
                                "PhysicalSizeX": x_um_ds,
                                "PhysicalSizeY": x_um_ds,
                                "PhysicalSizeXUnit": "µm",
                                "PhysicalSizeYUnit": "µm",
                            }
                    except Exception:
                        pass

                base = f"{lif_name}_img{i}_ds{downsample}_z{z0:02d}"
                img_path = img_out_dir / f"{base}.ome.tif"
                lbl_path = lbl_out_dir / f"{base}_label.ome.tif"

                tiff.imwrite(str(img_path), best2d, ome=True, metadata=meta)
                tiff.imwrite(str(lbl_path), np.zeros(best2d.shape, dtype=lbl_dtype), ome=True, metadata=meta)

                n_saved += 1

            except Exception:
                n_skipped += 1

    except Exception as e:
        return {"lif": lif_name, "saved": n_saved, "skipped": n_skipped, "error": repr(e)}

    finally:
        if lif is not None:
            try:
                lif.close()
            except Exception:
                pass

    return {"lif": lif_name, "saved": n_saved, "skipped": n_skipped}



# -------------------------
# run (MUST be guarded on Windows)
# -------------------------
if __name__ == "__main__":
    folder = Path(r"Z:\Bel\Vascumap_Example_Lifs\EXTRA_LIFS_PLEASE_ADD_HERE\lifs")
    lif_files = list(folder.glob("*.lif"))

    img_out_dir = Path(r"Z:\Bel\Vascumap_Example_Lifs\labelling_on_infocus_plane\images")
    lbl_out_dir = Path(r"Z:\Bel\Vascumap_Example_Lifs\labelling_on_infocus_plane\labels")
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    N_JOBS = 6
    DOWNSAMPLE = 4
    SIGMA_BG_DS = 10

    with tqdm_joblib(tqdm(desc="Processing LIFs", total=len(lif_files))) as _:
        results = Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(process_lif_file)(
                str(lif_path),
                str(img_out_dir),
                str(lbl_out_dir),
                downsample=DOWNSAMPLE,
                sigma_bg_ds=SIGMA_BG_DS,
                label_dtype="uint8",
            )
            for lif_path in lif_files
        )

    print("Done.")
    print(f"Total saved:   {sum(r.get('saved', 0) for r in results)}")
    print(f"Total skipped: {sum(r.get('skipped', 0) for r in results)}")

    errors = [r for r in results if "error" in r]
    if errors:
        print(f"\nErrors in {len(errors)} LIF(s):")
        for r in errors:
            print(f"  {r['lif']}: {r['error']}")

    for r in sorted(results, key=lambda x: x["lif"]):
        print(f"{r['lif']}: saved={r.get('saved',0)}, skipped={r.get('skipped',0)}")

Processing LIFs: 100%|██████████| 20/20 [1:03:10<00:00, 189.54s/it]

Done.
Total saved:   331
Total skipped: 41
20231130_23yo_cs_hmvec_day 7_merged_upd: saved=5, skipped=0
20250619_vascumap_dev25_11: saved=8, skipped=0
cropping_dorota: saved=1, skipped=0
defne_placenta_270325: saved=10, skipped=0
defne_placenta_270325_2: saved=14, skipped=0
defne_placenta_chip_big: saved=1, skipped=0
defne_placenta_chip_small: saved=1, skipped=0
empty_2026_01_13: saved=11, skipped=11
marina_m4_2024_10_24_fl2_fl6: saved=24, skipped=6
marina_m4_2024_11_14_mcl1_mcl3: saved=24, skipped=0
marina_m4_2024_12_04_fl4_fl12: saved=23, skipped=0
marina_m4_2025_02_05_mcl2_mcl4: saved=24, skipped=0
marina_m4_2025_02_28_fl32_fl33: saved=23, skipped=0
marina_m4_2025_03_23_mcl7_mcl8: saved=24, skipped=0
marina_m4_2025_06_27_fl41_fl43: saved=25, skipped=0
marina_m7_2025_10_06_fl33: saved=20, skipped=0
marina_m7_2025_10_22_fl12: saved=25, skipped=0
marina_m7_2025_11_12_fl43: saved=24, skipped=24
marina_m7_2025_12_05_fl39: saved=24, skipped=0
merged_stellaris_dorota: saved=20, skipped=0
